In [58]:
# =============================================================================
# Fig3 
# Row a: Deaths SUM + min/max band (no total line)
# Row b: Mortality rate MEAN ± SE (no total line)
# Row c: Decomposition waterfall (3 periods, background colour)
# Row d: 3 subplots — one per drive factor, 3 periods × East/West boxplot
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import geopandas as gpd

# ── 1. Paths ──────────────────────────────────────────────────────────────────

CSV_FILES = {
    2020: Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2020.csv"),
    2030: Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2030.csv"),
    2040: Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2040.csv"),
    2050: Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2050.csv"),
    2060: Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/earlypeak_NZ_CL_2060.csv"),
}
SHP_PATH = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
OUTFILE  = Path("/Users/shirley/Desktop/plots_V2/Fig3.png")

# ── 2. Config ─────────────────────────────────────────────────────────────────

AGE_BINS   = [25, 45, 65, 85, 200]
AGE_LABELS = ["25-44", "45-64", "65-84", "≥85"]
AGE_COLORS = ["#3C5488", "#4DBBD5", "#F39B7F", "#ca0020"]

DISEASE_LABELS = {
    "Chronic Obstructive Pulmonary Disease": "COPD",
    "Ischaemic Heart Disease":               "IHD",
    "Lower Respiratory Infections":          "LRI",
    "Lung Cancer":                           "Lung Cancer",
    "Strokes":                               "Stroke",
}

COL_AGE  = "#ca0020"
COL_POP  = "#4db3b3"
COL_RISK = "#bababa"

PERIODS  = ["2020-2040", "2040-2060", "2020-2060"]

PERIOD_BG = {
    "2020-2040": "#FFF3E0",
    "2040-2060": "#E8F5E9",
    "2020-2060": "#E3F2FD",
}
PERIOD_BG_SPAN = {
    "2020-2040": (-0.5, 1.7),
    "2040-2060": (2.3,  4.7),
    "2020-2060": (5.3,  7.5),
}
PERIOD_X_POS = {
    "2020-2040": {"East": 0, "West": 1},
    "2040-2060": {"East": 3, "West": 4},
    "2020-2060": {"East": 6, "West": 7},
}
PERIOD_COLORS = {
    "2020-2040": "#E8A838",
    "2040-2060": "#3DAA5C",
    "2020-2060": "#3A7EC6",
}

# x positions within each dist subplot (East/West per period)
DIST_X = {
    "2020-2040": {"East": 0.0,  "West": 0.55},
    "2040-2060": {"East": 1.6,  "West": 2.15},
    "2020-2060": {"East": 3.2,  "West": 3.75},
}
DIST_SPAN = {
    "2020-2040": (-0.32, 0.87),
    "2040-2060": (1.28,  2.47),
    "2020-2060": (2.88,  4.07),
}

PERIOD_FONT = 5   # unified period label fontsize for c and d

# ── 3. East/West (WGS84) ─────────────────────────────────────────────────────

HHY = {"lon": [127.5, 98.5], "lat": [50.2, 25.0]}

def hhy_lon_at_lat(lat):
    t = (lat - HHY["lat"][1]) / (HHY["lat"][0] - HHY["lat"][1])
    return HHY["lon"][1] + t * (HHY["lon"][0] - HHY["lon"][1])

city_shp = gpd.read_file(SHP_PATH)
cx = city_shp.geometry.centroid.x
cy = city_shp.geometry.centroid.y
city_shp["region"] = np.where(cx > hhy_lon_at_lat(cy), "East", "West")
region_map = city_shp[["English", "region"]].rename(columns={"English": "city"})

# ── 4. Load data ──────────────────────────────────────────────────────────────

def load_year(year, path):
    df = pd.read_csv(path)
    df["age_group"] = pd.cut(df["age"], bins=AGE_BINS,
                             labels=AGE_LABELS, right=False)
    df["year"] = year
    return df

all_df = pd.concat([load_year(y, p) for y, p in CSV_FILES.items()],
                   ignore_index=True)
all_df = all_df.merge(region_map, on="city", how="left")

# ── 5. Row a aggregation ──────────────────────────────────────────────────────

city_agg_a = (
    all_df
    .groupby(["year","city","region","disease","age_group"], observed=True)
    .agg(deaths    =("mo_total",    "sum"),
         deaths_min=("mo_total_min","sum"),
         deaths_max=("mo_total_max","sum"))
    .reset_index()
)
sum_agg = (
    city_agg_a
    .groupby(["year","region","disease","age_group"], observed=True)
    .agg(deaths_sum=("deaths","sum"),
         deaths_lo =("deaths_min","sum"),
         deaths_hi =("deaths_max","sum"))
    .reset_index()
)

# ── 6. Row b aggregation ──────────────────────────────────────────────────────

city_agg_b = (
    all_df
    .groupby(["year","city","region","disease","age_group"], observed=True)
    .agg(deaths=("mo_total","sum"), pop=("pop","sum"))
    .reset_index()
)
city_agg_b["rate"] = city_agg_b["deaths"] / city_agg_b["pop"] * 100_000

def se(x):
    return x.std() / np.sqrt(len(x))

mean_agg = (
    city_agg_b
    .groupby(["year","region","disease","age_group"], observed=True)
    .agg(rate_mean=("rate","mean"), rate_se=("rate",se))
    .reset_index()
)
mean_agg["rate_upper"] = mean_agg["rate_mean"] + mean_agg["rate_se"]
mean_agg["rate_lower"] = mean_agg["rate_mean"] - mean_agg["rate_se"]

DISEASES = list(DISEASE_LABELS.keys())
YEARS    = sorted(CSV_FILES.keys())

# ── 7. Decomposition (LMDI) ───────────────────────────────────────────────────

df20 = pd.read_csv(CSV_FILES[2020])
df40 = pd.read_csv(CSV_FILES[2040])
df60 = pd.read_csv(CSV_FILES[2060])

def log_mean(a, b):
    if abs(a - b) < 1e-10: return a
    if a <= 0 or b <= 0:   return 0
    return (a - b) / (np.log(a) - np.log(b))

def decompose_all(df_t0, df_t1, period):
    results = []
    for city in df_t0["city"].unique():
        d0 = (df_t0[df_t0["city"]==city]
              .groupby("age").agg(pop=("pop","sum"),D=("mo_total","sum"))
              .reset_index())
        d1 = (df_t1[df_t1["city"]==city]
              .groupby("age").agg(pop=("pop","sum"),D=("mo_total","sum"))
              .reset_index())
        d  = d0.merge(d1, on="age", suffixes=("_0","_1"))
        mask = (d["pop_0"]>0)&(d["pop_1"]>0)
        d  = d[mask].copy()
        if len(d)==0: continue
        N0=d["pop_0"].sum(); N1=d["pop_1"].sum()
        s0=d["pop_0"]/N0;   s1=d["pop_1"]/N1
        r0=d["D_0"]/d["pop_0"]; r1=d["D_1"]/d["pop_1"]
        Di0=N0*s0*r0; Di1=N1*s1*r1
        w=np.array([log_mean(a,b) for a,b in zip(Di1,Di0)])
        results.append({
            "city":city,"period":period,
            "delta":       Di1.sum()-Di0.sum(),
            "pop_effect":  np.sum(w*np.log(N1/N0)),
            "age_effect":  np.sum(w*np.log(np.where(s0>0,s1/s0,1))),
            "risk_effect": np.sum(w*np.log(np.where(r0>0,r1/r0,1))),
        })
    return pd.DataFrame(results)

decomp = pd.concat([
    decompose_all(df20,df40,"2020-2040"),
    decompose_all(df40,df60,"2040-2060"),
    decompose_all(df20,df60,"2020-2060"),
], ignore_index=True)
decomp = decomp.merge(region_map, on="city", how="left")

decomp_region = (
    decomp.groupby(["period","region"])
    .agg(pop_effect=("pop_effect","sum"), age_effect=("age_effect","sum"),
         risk_effect=("risk_effect","sum"), delta=("delta","sum"))
    .reset_index()
)

# ── 8. Global style ───────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.labelsize":   6,
    "axes.titlesize":   7,
    "axes.titleweight": "bold",
    "xtick.labelsize":  5,
    "ytick.labelsize":  5,
})

W_IN, H_IN, DPI = 18/2.54, 18/2.54, 400

# ── 9. Layout ─────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(W_IN, H_IN), dpi=DPI)
gs_outer = gridspec.GridSpec(
    nrows=3, ncols=1, figure=fig,
    height_ratios=[1, 1, 1.1],
    hspace=0.3,
)

gs_a = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=gs_outer[0], wspace=0.3)
axes_a = [fig.add_subplot(gs_a[0,c]) for c in range(5)]

gs_b = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=gs_outer[1], wspace=0.3)
axes_b = [fig.add_subplot(gs_b[0,c]) for c in range(5)]

gs_cd = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_outer[2],
    width_ratios=[0.6, 1.8], wspace=0.25,
)
ax_wf = fig.add_subplot(gs_cd[0, 0])

gs_d = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_cd[0, 1], wspace=0.35)
axes_d = [fig.add_subplot(gs_d[0, c]) for c in range(3)]

# ── 10. Row a: Deaths SUM + band ─────────────────────────────────────────────

for col, disease in enumerate(DISEASES):
    ax = axes_a[col]
    data_d = sum_agg[sum_agg["disease"]==disease]

    for age, color in zip(AGE_LABELS, AGE_COLORS):
        data_a = data_d[data_d["age_group"]==age]
        for region, ls in [("East","-"),("West","--")]:
            sub = data_a[data_a["region"]==region].sort_values("year")
            if len(sub)==0: continue
            ax.plot(sub["year"], sub["deaths_sum"],
                    color=color, ls=ls, lw=1.0, marker="o", ms=2, zorder=3)
            ax.fill_between(sub["year"], sub["deaths_lo"], sub["deaths_hi"],
                            color=color, alpha=0.12, lw=0, zorder=2)

    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["bottom","left"]].set_linewidth(0.4)
    ax.tick_params(width=0.4, length=2)
    ax.set_xticks(YEARS)
    ax.set_xticklabels([str(y) for y in YEARS], rotation=30, ha="right")
    ax.set_xlim(2018, 2062)
    ax.set_title(DISEASE_LABELS[disease], fontsize=7, pad=3)
    if col==0:
        ax.text(-0.22, 1.05, "a", transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="bottom")
        ax.set_ylabel("Deaths", fontsize=6)

for col in range(len(DISEASES)):
    ax = axes_a[col]
    mv = ax.get_ylim()[1]
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x,_: f"{x/1000:.0f}K" if mv>=1000 else f"{x:.0f}"
        )
    )

# ── 11. Row b: Rate MEAN ± SE ─────────────────────────────────────────────────

for col, disease in enumerate(DISEASES):
    ax = axes_b[col]
    data_d = mean_agg[mean_agg["disease"]==disease]

    for age, color in zip(AGE_LABELS, AGE_COLORS):
        data_a = data_d[data_d["age_group"]==age]
        for region, ls in [("East","-"),("West","--")]:
            sub = data_a[data_a["region"]==region].sort_values("year")
            if len(sub)==0: continue
            ax.plot(sub["year"], sub["rate_mean"],
                    color=color, ls=ls, lw=1.0, marker="o", ms=2, zorder=3)
            ax.fill_between(sub["year"], sub["rate_lower"], sub["rate_upper"],
                            color=color, alpha=0.12, lw=0, zorder=2)

    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["bottom","left"]].set_linewidth(0.4)
    ax.tick_params(width=0.4, length=2)
    ax.set_xticks(YEARS)
    ax.set_xticklabels([str(y) for y in YEARS], rotation=30, ha="right")
    ax.set_xlim(2018, 2062)
    ax.set_title(DISEASE_LABELS[disease], fontsize=7, pad=3)
    if col==0:
        ax.text(-0.22, 1.05, "b", transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="bottom")
        ax.set_ylabel("Mortality rate (per 100k)", fontsize=6)

# ── 12. Row c: Waterfall ─────────────────────────────────────────────────────

ax = ax_wf
ax.text(-0.14, 1.05, "c", transform=ax.transAxes,
        fontsize=8, fontweight="bold", va="bottom")

for period in PERIODS:
    x0, x1 = PERIOD_BG_SPAN[period]
    ax.axvspan(x0, x1, color=PERIOD_BG[period], alpha=0.6, zorder=0, lw=0)
    ax.text((x0+x1)/2, 1.01, period,
            transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=PERIOD_FONT,
            fontweight="bold", color="grey")

    data = decomp_region[decomp_region["period"]==period]
    for region in ["East","West"]:
        sub = data[data["region"]==region]
        if len(sub)==0: continue
        x         = PERIOD_X_POS[period][region]
        age_m     = sub["age_effect"].values[0]
        pop_m     = sub["pop_effect"].values[0]
        risk_m    = sub["risk_effect"].values[0]
        net_m     = sub["delta"].values[0]
        total_abs = abs(age_m)+abs(pop_m)+abs(risk_m)

        pos_btm = 0.0
        for val, col_ in [(age_m,COL_AGE),(max(pop_m,0),COL_POP)]:
            if val > 1e-6:
                ax.bar(x, val, bottom=pos_btm, width=0.8,
                       color=col_, lw=0, zorder=3)
                pct = abs(val)/total_abs*100
                if pct > 15:
                    ax.text(x, pos_btm+val/2, f"{pct:.0f}%",
                            ha="center", va="center", fontsize=4,
                            color="white", fontweight="bold", zorder=5)
                pos_btm += val

        neg_btm = 0.0
        for val, col_ in [(risk_m,COL_RISK),(min(pop_m,0),COL_POP)]:
            if val < -1e-6:
                ax.bar(x, val, bottom=neg_btm, width=0.8,
                       color=col_, lw=0, zorder=3)
                pct = abs(val)/total_abs*100
                if pct > 5:
                    ax.text(x, neg_btm+val/2, f"{pct:.0f}%",
                            ha="center", va="center", fontsize=4,
                            color="white", fontweight="bold", zorder=5)
                neg_btm += val

        marker = "D" if region=="East" else "o"
        ax.plot(x, net_m, marker=marker, ms=2,
                color="black", zorder=4, lw=0)

ax.set_xticks([0,1,3,4,6,7])
ax.set_xticklabels(["East","West"]*3, fontsize=5, rotation=30)
ax.set_xlim(-0.6, 7.6)
ax.axhline(0, color="grey", lw=0.3, ls="--", zorder=1)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1000:.0f}K"))
ax.set_ylabel("Change in deaths", fontsize=6,labelpad=1)
ax.spines[["top","right"]].set_visible(False)
ax.spines[["bottom","left"]].set_linewidth(0.4)
ax.tick_params(width=0.4, length=2, pad=1)

# ── 13. Row d: 3 subplots, one per drive factor ───────────────────────────────

# ← 颜色来自 FACTORS tuple，不被 PERIOD_COLORS 覆盖
FACTORS = [
    ("age_effect",  "Ageing effect (deaths)",          COL_AGE),
    ("pop_effect",  "Population size effect (deaths)", COL_POP),   
    ("risk_effect", "PM$_{2.5}$ risk effect (deaths)", COL_RISK),
]
rng = np.random.default_rng(42)
BW_D = 0.15

for fi, (factor, ylabel, fac_color) in enumerate(FACTORS):   # ← fac_color 不再被覆盖
    ax = axes_d[fi]

    if fi == 0:
        ax.text(-0.28, 1.05, "d", transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="bottom")

    ax.set_ylabel(ylabel, fontsize=6, labelpad=1)

    for period in PERIODS:
        # 背景色与 c 图一致
        x0, x1 = DIST_SPAN[period]
        ax.axvspan(x0, x1, color=PERIOD_BG[period], alpha=0.6, zorder=0, lw=0)
        ax.text((x0+x1)/2, 1.01, period,
                transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=PERIOD_FONT,   # ← 与 c 统一
                fontweight="bold", color="grey")

        # ← 用因子颜色，不用 PERIOD_COLORS
        for region, ls in [("East", "-"), ("West", "--")]:
            sub = (decomp[(decomp["period"] == period) &
                          (decomp["region"] == region)][factor].dropna())
            sub = sub[np.isfinite(sub)]
            if len(sub) < 2:
                continue

            x = DIST_X[period][region]
            q1, med, q3 = np.percentile(sub, [25, 50, 75])
            iqr = q3 - q1
            lo  = sub[sub >= q1 - 1.5*iqr].min()
            hi_ = sub[sub <= q3 + 1.5*iqr].max()

            ax.scatter(x + rng.uniform(-0.07, 0.07, len(sub)), sub,
                       color=fac_color, s=1.0, alpha=0.15, lw=0, zorder=2)
            rect = mpatches.FancyBboxPatch(
                (x-BW_D, q1), 2*BW_D, q3-q1,
                boxstyle="square,pad=0",
                facecolor="white", edgecolor=fac_color,
                lw=0.8, ls=ls, alpha=0.9, zorder=3)
            ax.add_patch(rect)
            ax.plot([x-BW_D, x+BW_D], [med, med],
                    color=fac_color, lw=1.0, ls=ls, zorder=4)
            ax.plot([x, x], [lo, q1],  color=fac_color, lw=0.5, ls=ls, zorder=3)
            ax.plot([x, x], [q3, hi_], color=fac_color, lw=0.5, ls=ls, zorder=3)
            cw = 0.05
            for yy in [lo, hi_]:
                ax.plot([x-cw, x+cw], [yy, yy],
                        color=fac_color, lw=0.5, ls=ls, zorder=3)

    # East/West x 轴刻度标签
    xtick_pos, xtick_lab = [], []
    for period in PERIODS:
        for region in ["East", "West"]:
            xtick_pos.append(DIST_X[period][region])
            xtick_lab.append("East" if region == "East" else "West")

    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_lab, fontsize=5, color="black", rotation=30)   
    ax.tick_params(axis="x", width=0.4, length=2,pad=1)      

    ax.axhline(0, color="grey", lw=0.3, ls="--", zorder=1)
    ax.set_xlim(DIST_SPAN["2020-2040"][0] - 0.1, DIST_SPAN["2020-2060"][1] + 0.1)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K")
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["bottom", "left"]].set_linewidth(0.4)
    ax.tick_params(axis="y", width=0.4, length=2)

# ── 14. Legend ────────────────────────────────────────────────────────────────

age_handles = [
    mlines.Line2D([],[],color=c,lw=1.2,marker="o",ms=2.5,label=a)
    for a,c in zip(AGE_LABELS, AGE_COLORS)
]
ew_handles = [
    mlines.Line2D([],[],color="grey",lw=1.2,ls="-",  label="East of Hu Line"),
    mlines.Line2D([],[],color="grey",lw=1.2,ls="--", label="West of Hu Line"),
]
decomp_handles = [
    mpatches.Patch(color=COL_AGE,  label="Ageing"),
    mpatches.Patch(color=COL_POP,  label="Population size"),
    mpatches.Patch(color=COL_RISK, label="PM$_{2.5}$ risk"),
]


panel_ab = mlines.Line2D([],[],color="none",label="Panel a-b:")
panel_cd = mlines.Line2D([],[],color="none",label="Panel c-d:")

row1_handles = [panel_ab] + age_handles + ew_handles
row2_handles = [panel_cd] + decomp_handles 

leg1 = fig.legend(
    handles=row1_handles,
    loc="lower center", ncol=len(row1_handles),
    fontsize=6, frameon=False,
    bbox_to_anchor=(0.5, 0.03),
    handlelength=1.6, columnspacing=0.9,
)
leg1.get_texts()[0].set_fontweight("bold")

leg2 = fig.legend(
    handles=row2_handles,
    loc="lower center", ncol=len(row2_handles),
    fontsize=6, frameon=False,
    bbox_to_anchor=(0.5, 0.01),
    handlelength=1.6, columnspacing=0.9,
)
leg2.get_texts()[0].set_fontweight("bold")
fig.add_artist(leg1)

fig.tight_layout(rect=[0, 0.07, 1, 1])

# ── 15. Save ──────────────────────────────────────────────────────────────────

OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"✓ Saved → {OUTFILE}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_39414/3849217433.py:95: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cx = city_shp.geometry.centroid.x
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_39414/3849217433.py:96: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cy = city_shp.geometry.centroid.y
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_39414/3849217433.py:497: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0.07, 1, 1])


✓ Saved → /Users/shirley/Desktop/plots_V2/Fig3.png
